## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-learn`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Two fraud-detection models: ROC curves compared

**ROC Curve** · a worked example from the StatNex teaching guide

**The story.** A bank tests two fraud-detection models on 18 card payments, 10 of which turned out to be fraud. Each model gives every payment a suspicion score, where higher means more likely to be fraud.

**The question.** Which model separates fraud from genuine payments better, and where should the alarm threshold be set?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `10_control_chart/roc_curve.csv` (18 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Model A score,Model B score,Fraud
12.0,20.0,0.0
18.0,35.0,0.0
25.0,28.0,0.0
31.0,50.0,0.0
22.0,42.0,0.0
40.0,61.0,0.0
15.0,33.0,0.0
55.0,45.0,0.0
48.0,40.0,1.0
62.0,55.0,1.0
70.0,38.0,1.0
58.0,62.0,1.0
81.0,48.0,1.0
66.0,70.0,1.0
90.0,52.0,1.0
74.0,66.0,1.0
45.0,30.0,1.0
85.0,58.0,1.0
""")

## Helper: delong

A step that no library does, written out. In the code download it is the file `delong.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""DeLong's placement values: the area under an ROC curve and what its variance needs.

No library in this kit computes them (in R the pROC package does). Each positive case is
compared with every negative one; the comparisons give the area and, case by case, the
"placements" whose spread is DeLong's standard error.
"""

import numpy as np


def placements(positive, negative):
    """(AUC, V10, V01).

    V10[i] is the share of negatives that positive case i out-scores, V01[j] the share of
    positives that out-score negative case j; a tie counts one half. Both average to the
    AUC, the probability that a random positive scores above a random negative.
    """
    beats = (positive[:, None] > negative[None, :]) + 0.5 * (positive[:, None] == negative[None, :])
    return beats.mean(), beats.mean(axis=1), beats.mean(axis=0)

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
# placements: defined in the helper cell delong above (delong.py in the code download)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

## The data

In [ ]:
# One column per test and a two-valued outcome column. Each test uses the cases where both
# it and the outcome are recorded.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
tests = ["Model A score", "Model B score"]
outcome = "Fraud"
confidence = 95 / 100
z = stats.norm.ppf(0.5 + confidence / 2)

## Which value is "disease present"

In [ ]:
# Every number below depends on which outcome value counts as a case of the disease: the
# one [Value meaning disease present] names or, left blank, the larger of two numbers. The
# other value is "disease absent". Check it before reading on.
positive_label = 1
negative_label = 0
print(f"Disease present: {positive_label!r}    disease absent: {negative_label!r}")


def split(frame, test):
    """The test's values for the diseased (positive) and healthy (negative) cases."""
    values = pd.to_numeric(frame[test], errors="coerce")
    usable = values.notna() & frame[outcome].notna()
    return (values[usable & (frame[outcome] == positive_label)].to_numpy(float),
            values[usable & (frame[outcome] == negative_label)].to_numpy(float))


groups = {test: split(data, test) for test in tests}

## Descriptive statistics

In [ ]:
# Each test's values in the healthy and the diseased group. A test can only tell the
# groups apart as far as these two distributions are apart.
described = {}
for test, (positive, negative) in groups.items():
    for label, sample in ((negative_label, negative), (positive_label, positive)):
        described[f"{test} / {label}"] = [len(sample), sample.mean(), sample.var(ddof=1),
                                          sample.std(ddof=1), sample.min(), sample.max()]
descriptive = pd.DataFrame(described, index=["n", "mean", "variance", "SD", "min", "max"])
print(descriptive)



def labelled(positive, negative):
    """Each case's outcome (1 = disease) and its test value, as scikit-learn takes them."""
    return (np.r_[np.ones(len(positive)), np.zeros(len(negative))],
            np.r_[positive, negative])

## Area under the ROC curve

In [ ]:
# The area (AUC, scikit-learn's roc_auc_score) is the probability that a randomly chosen
# diseased case scores higher than a randomly chosen healthy one (a tie counts one half):
# 0.5 is a coin toss, 1 a perfect test. Its standard error is DeLong's, from the
# "placements" in delong.py -- neither SciPy nor statsmodels nor scikit-learn has it. It
# assumes nothing about the shape of the two distributions. The interval is
# area +/- z x SE, cut at 0 and 1.
# The chi-square tests "this test does no better than chance" (area = 0.5). It divides
# by the area's variance under that hypothesis, ((N + 1) - sum(t^3 - t) / (N(N - 1)))
# / (12 m n), with t the size of each group of tied values: this is the Mann-Whitney
# (Wilcoxon rank-sum) test, and its P value is scipy's mannwhitneyu without the
# continuity correction. It assumes only that the cases are independent.


def null_variance(positive, negative):
    """The area's variance if the test were no better than chance (ties corrected)."""
    m, n = len(positive), len(negative)
    _, counts = np.unique(np.r_[positive, negative], return_counts=True)
    ties = np.sum(counts.astype(float) ** 3 - counts) / ((m + n) * (m + n - 1))
    return (m + n + 1 - ties) / (12 * m * n)


rows = []
for test, (positive, negative) in groups.items():
    area = roc_auc_score(*labelled(positive, negative))
    _, v10, v01 = placements(positive, negative)  # the placements also average to the area
    se = np.sqrt(v10.var(ddof=1) / len(positive) + v01.var(ddof=1) / len(negative))
    spread = null_variance(positive, negative)
    chi2 = (area - 0.5) ** 2 / spread if spread > 0 else np.nan
    p = (stats.mannwhitneyu(positive, negative, use_continuity=False,
                            method="asymptotic").pvalue if spread > 0 else np.nan)
    # An area lies in [0, 1], so the interval is cut there.
    rows.append([area, se, max(area - z * se, 0.0), min(area + z * se, 1.0), chi2, 1, p])
auc = pd.DataFrame(rows, index=tests, columns=["area", "SE", "lower", "upper", "chi-square",
                                               "df", "P"])
print()
print(auc)

## Comparison of areas (DeLong, paired)

In [ ]:
# Two tests measured on the same cases are correlated, so their areas cannot be
# compared as if independent: the covariance of their placements enters the variance of
# the difference (DeLong, DeLong and Clarke-Pearson, 1988). Only cases with both tests
# recorded are used. The chi-square on 1 df tests "the two areas are equal".
rows = []
for first, second in combinations(tests, 2):
    both = data[data[[first, second]].apply(pd.to_numeric, errors="coerce").notna().all(axis=1)]
    pair = [split(both, test) for test in (first, second)]
    (area1, v10_1, v01_1), (area2, v10_2, v01_2) = [placements(p, n) for p, n in pair]
    s10 = np.cov(np.vstack([v10_1, v10_2]))
    s01 = np.cov(np.vstack([v01_1, v01_2]))
    variance = ((s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / len(pair[0][0])
                + (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / len(pair[0][1]))
    difference, se = area1 - area2, np.sqrt(variance)
    chi2 = (difference / se) ** 2 if se > 0 else np.nan
    rows.append([first, second, difference, se, difference - z * se, difference + z * se,
                 chi2, 1, stats.chi2.sf(chi2, 1)])
comparison = pd.DataFrame(rows, columns=["test 1", "test 2", "difference", "SE", "lower",
                                         "upper", "chi-square", "df", "P"])
print()
print(comparison.to_string(index=False))

## ROC curve data

In [ ]:
# scikit-learn's roc_curve gives the false positive fraction (FPF, 1 - specificity) and
# the true positive fraction (TPF, the sensitivity) with every observed value as a
# cut-off; a case is called positive at or above it. It lists the cut-offs from the top
# down and marks the point where nobody is called positive, (0, 0), with an infinite
# cut-off; the app lists them from the bottom up and puts the largest value + 1 there.
# The diagnostic odds ratio, (TPF / (1 - TPF)) / (FPF / (1 - FPF)), is how many times
# higher the odds of a positive call are for a diseased case than for a healthy one; it
# is undefined (NaN here, "-" in the app) where TPF = 1, FPF = 0 or FPF = 1.
curves = {}
for test, (positive, negative) in groups.items():
    truth, values = labelled(positive, negative)
    fpf, tpf, cutoffs = roc_curve(truth, values, drop_intermediate=False)
    cutoffs[0] = values.max() + 1
    fpf, tpf, cutoffs = fpf[::-1], tpf[::-1], cutoffs[::-1]
    defined = (tpf < 1) & (fpf > 0) & (fpf < 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        odds = np.where(defined, (tpf / (1 - tpf)) / (fpf / (1 - fpf)), np.nan)
    curves[test] = pd.DataFrame({"cut-off": cutoffs, "FPF": fpf, "TPF": tpf, "odds ratio": odds})
points = pd.concat(curves, axis=1)
print(points)

## Best cut-offs

In [ ]:
# Two common ways to pick one row of the curve. Closest to the top-left corner (0, 1):
# the smallest FPF^2 + (1 - TPF)^2. Furthest above the diagonal: the largest Youden index
# J = TPF - FPF. Both weigh a missed case and a false alarm equally; when the two errors
# cost different amounts, neither is the right cut-off. The first row wins a tie.
closest = pd.concat({test: curve.iloc[[(curve["FPF"] ** 2 + (1 - curve["TPF"]) ** 2).argmin()]]
                     .reset_index(drop=True) for test, curve in curves.items()}, axis=1)
farthest = pd.concat({test: curve.iloc[[(curve["TPF"] - curve["FPF"]).argmax()]]
                      .reset_index(drop=True) for test, curve in curves.items()}, axis=1)
print("\nClosest to (0, 1):")
print(closest)
print("\nFurthest from the diagonal:")
print(farthest)

## What the result says

The area under the ROC curve (AUC) is the chance that a model gives a fraud a higher score than a genuine payment. Model A has an area of 0.975, with a confidence interval (the range of plausible values) from 0.916 to 1. Model B has 0.7625, with an interval from 0.53 to 0.99.

The difference in area is 0.2125, but its P value is 0.054 and its confidence interval runs from -0.004 to 0.429. A P value is the chance of a difference at least this large if the models were really equal. So 18 payments do not pin down the size of the gap.

For Model A, a cut-off of 45 catches every fraud (TPF = 1) and wrongly flags 0.125 of genuine payments (FPF). Model A looks far better, but the alarm threshold is a separate decision.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.